<div style="display: flex; background-color: RGB(255,114,0);">
<h1 style="margin: auto; padding: 30px; ">ANALYSE DU STOCK ET DES VENTES DU SITE BOTTLENECK — VERSION 2 AMÉLIORÉE</h1>
</div>

# OBJECTIF DE CE NOTEBOOK (v2 — Projet 13)

Ce notebook est la **version améliorée** du livrable du projet 6, réalisée dans le cadre du projet 13 (amélioration critique et documentée d'un livrable existant, appuyée par une veille technologique).

**Ce qui change par rapport à la v1 :**

| # | Amélioration | Section |
|---|---|---|
| 1 | Validation **automatique** de la qualité des données avec **Pandera** (règles déclaratives) au lieu de corrections « en dur » | 3 et 5 |
| 2 | Détection des prix aberrants **comparée** : z-score (v1) vs **IQR** vs **Isolation Forest** multivariée | 6 et 7 |
| 3 | **Reproductibilité** : environnement figé, seed fixée, narration structurée, ancres de validation | 1 et 11 |
| 4 | **Performance mesurée** : pandas vs **Polars** vs **DuckDB** chronométrés sur les jointures réelles | 8 |

**Décisions issues de la veille (voir journal des tests IA) :**
- *Pandera* retenu face à Great Expectations (plus léger, schéma dans le code, adapté à un notebook unique ; GX reste pertinent si industrialisation future).
- *IQR* retenu comme détecteur robuste de premier niveau ; *Isolation Forest* utilisé en complément multivarié (prix × prix d'achat × stock × ventes).
- *pandas* conservé face à Polars et DuckDB : décision fondée sur un **benchmark réel en section 8**, pas sur une intuition.

**Sommaire :** 1. Environnement · 2. Chargement · 3. Validation qualité (données brutes) · 4. Nettoyage documenté · 5. Fusion + validation finale · 6. Prix : z-score vs IQR · 7. Isolation Forest · 8. Performance : pandas vs Polars vs DuckDB · 9. Analyses métier (CA, marges, rotation, ABC, flags) · 10. Synthèse et recommandations · 11. Reproductibilité

<div style="background-color: RGB(51,165,182);">
<h2 style="margin: auto; padding: 20px; color:#fff; ">1 — Environnement et reproductibilité</h2>
</div>

Les versions des bibliothèques sont affichées et figées dans `requirements.txt`. La seed aléatoire est fixée une seule fois, ici, et utilisée par tout algorithme stochastique (Isolation Forest, section 7).

In [1]:
import sys
import numpy as np
import pandas as pd
import pandera.pandas as pa
import plotly.express as px
from sklearn.ensemble import IsolationForest

SEED = 42  # seed unique pour toute étape aléatoire

print("python   :", sys.version.split()[0])
print("pandas   :", pd.__version__)
print("numpy    :", np.__version__)
import importlib.metadata
print("pandera  :", importlib.metadata.version("pandera"))
import sklearn, plotly
print("sklearn  :", sklearn.__version__)
print("plotly   :", plotly.__version__)

python   : 3.12.15
pandas   : 3.0.2
numpy    : 2.4.4
pandera  : 0.35.0
sklearn  : 1.9.0
plotly   : 7.1.0


<div style="background-color: RGB(51,165,182);">
<h2 style="margin: auto; padding: 20px; color:#fff; ">2 — Chargement des fichiers</h2>
</div>

Trois fichiers sources, **jamais modifiés** (principe de traçabilité) :
- `erp.csv` : catalogue produits (prix, prix d'achat, stock) ;
- `web.csv` : ventes en ligne (export WordPress) ;
- `liaison.csv` : correspondance `product_id` (ERP) ↔ `id_web` (web).

In [2]:
erp = pd.read_csv("data/erp.csv")
web = pd.read_csv("data/web.csv", low_memory=False)
liaison = pd.read_csv("data/liaison.csv")

for nom, df_ in [("erp", erp), ("web", web), ("liaison", liaison)]:
    print(f"{nom:8s} : {df_.shape[0]:5d} lignes x {df_.shape[1]:2d} colonnes")

erp      :   825 lignes x  6 colonnes
web      :  1513 lignes x 29 colonnes
liaison  :   825 lignes x  2 colonnes


<div style="background-color: RGB(51,165,182);">
<h2 style="margin: auto; padding: 20px; color:#fff; ">3 — Validation automatique de la qualité (données brutes)</h2>
</div>

**Nouveauté v2.** Au lieu de corriger « en dur » au fil des cellules, on déclare un **contrat de données** par fichier (règles explicites : types, unicité, valeurs autorisées, bornes) et on le fait vérifier automatiquement par **Pandera** à chaque exécution. Si un futur export ne respecte pas le contrat, le notebook s'arrête ici avec la liste précise des lignes fautives — au lieu de produire silencieusement des chiffres faux.

Règles volontairement « permissives » à ce stade : on valide la **structure** des fichiers bruts ; les règles métier strictes (prix > 0, stock ≥ 0…) sont vérifiées **après nettoyage** en section 5.

In [3]:
from pandera import Column, Check, DataFrameSchema

schema_erp_brut = DataFrameSchema({
    "product_id": Column(int, Check.gt(0), unique=True, nullable=False),
    "onsale_web": Column(int, Check.isin([0, 1])),
    "price": Column(str),          # brut : décimales à virgule, converties en section 4
    "stock_quantity": Column(int),
    "stock_status": Column(str, Check.isin(["instock", "outofstock"])),
    "purchase_price": Column(str),
})

schema_liaison = DataFrameSchema({
    "id_web": Column(str, nullable=True),   # texte : '13127-1', 'bon-cadeau-25-euros' sont des clés valides
    "product_id": Column(int, Check.gt(0), unique=True, nullable=False),
})

schema_web_brut = DataFrameSchema({
    "sku": Column(str, nullable=True),
    "total_sales": Column(float, nullable=True),
    "product_type": Column(str, nullable=True),
    "post_title": Column(str, nullable=True),
}, strict=False)  # le fichier web contient 29 colonnes, on n'en retient que 4

def valider(nom, schema, df_):
    """Valide un DataFrame et rapporte les échecs de façon lisible (toutes les erreurs, pas la première)."""
    try:
        schema.validate(df_, lazy=True)
        print(f"[OK] {nom} : {len(df_)} lignes conformes au contrat")
    except pa.errors.SchemaErrors as e:
        print(f"[ECHEC] {nom} : {len(e.failure_cases)} cas non conformes")
        display(e.failure_cases.head(10))

valider("erp brut", schema_erp_brut, erp)
valider("liaison", schema_liaison, liaison)
valider("web brut", schema_web_brut, web)

[OK] erp brut : 825 lignes conformes au contrat
[OK] liaison : 825 lignes conformes au contrat
[OK] web brut : 1513 lignes conformes au contrat


/tmp/ipykernel_13248/1017075853.py:3: FutureWarning: Importing pandas-specific classes and functions from the
top-level pandera module will be **removed in a future version of pandera**.
If you're using pandera to validate pandas objects, we highly recommend updating
your import:

```
# old import
import pandera as pa

# new import
import pandera.pandas as pa
```

If you're using pandera to validate objects from other compatible libraries
like pyspark or polars, see the supported libraries section of the documentation
for more information on how to import pandera:

https://pandera.readthedocs.io/en/stable/supported_libraries.html

To disable this warning, set the environment variable:

```
export DISABLE_PANDERA_IMPORT_WARNING=True
```

  schema_erp_brut = DataFrameSchema({


<div style="background-color: RGB(51,165,182);">
<h2 style="margin: auto; padding: 20px; color:#fff; ">4 — Nettoyage documenté</h2>
</div>

Chaque correction suit une **règle métier explicite**, et le nombre de lignes concernées est affiché (journalisation). Rien n'est corrigé « à la main » sur une ligne précise : les règles s'appliqueront telles quelles à un nouvel export.

| Règle | Correction | Justification |
|---|---|---|
| R1. Prix saisis avec virgule décimale | Conversion `,` → `.` puis float | Format français de l'export ERP |
| R2. Prix ou stocks négatifs | Prix → valeur absolue ; stock → 0 | Erreurs de saisie (un prix/stock négatif n'existe pas) |
| R3. `stock_status` incohérent avec la quantité | Recalcul depuis `stock_quantity` | La quantité physique prime sur l'étiquette |
| R4. Lignes web totalement vides | Suppression | Artefacts d'export (pièces jointes) |
| R5. Doublons complets dans le web | Suppression, garde la 1re occurrence | Même produit exporté plusieurs fois |
| R6. Doublons résiduels sur la clé `sku` | Garde la ligne **la plus renseignée** | Réduit la perte d'information |
| R7. Clés non numériques (`13127-1`, `bon-cadeau-25-euros`) | **Conservées telles quelles** | Ce sont des clés valides : elles existent aussi dans `liaison.id_web` |
| R8. Ventes négatives | Valeur absolue | Retours/annulations saisis en négatif |

In [4]:
# --- ERP : règles R1 à R3 ---
erp["price"] = erp["price"].str.replace(",", ".").astype(float)
erp["purchase_price"] = erp["purchase_price"].str.replace(",", ".").astype(float)

n_neg_price = int((erp["price"] < 0).sum())
n_neg_stock = int((erp["stock_quantity"] < 0).sum())
erp["price"] = erp["price"].abs()
erp["stock_quantity"] = erp["stock_quantity"].clip(lower=0)

incoh = int((((erp["stock_quantity"] == 0) & (erp["stock_status"] == "instock")) |
             ((erp["stock_quantity"] > 0) & (erp["stock_status"] == "outofstock"))).sum())
erp.loc[erp["stock_quantity"] == 0, "stock_status"] = "outofstock"
erp.loc[erp["stock_quantity"] > 0, "stock_status"] = "instock"

print(f"R1 : prix convertis en float (price, purchase_price)")
print(f"R2 : {n_neg_price} prix négatifs -> valeur absolue ; {n_neg_stock} stocks négatifs -> 0")
print(f"R3 : {incoh} incohérences stock_status corrigées")
erp.head(3)

R1 : prix convertis en float (price, purchase_price)
R2 : 3 prix négatifs -> valeur absolue ; 2 stocks négatifs -> 0
R3 : 2 incohérences stock_status corrigées


,product_id,onsale_web,price,stock_quantity,stock_status,purchase_price
0,3847,1,24.2,16,instock,12.88
1,3849,1,34.3,10,instock,17.54
2,3850,1,20.8,0,outofstock,10.64


In [5]:
# --- WEB : règles R4 à R8 ---
COLS_WEB = ["sku", "total_sales", "product_type", "post_title", "post_name"]
w = web[COLS_WEB].copy()

n_avant = len(w)
w = w.drop_duplicates(keep="first")                       # R5
mask_vides = w[COLS_WEB].isnull().all(axis=1)
w = w[~mask_vides]                                        # R4
print(f"R5 : {n_avant - len(w) - int(mask_vides.sum())} doublons complets supprimés")
print(f"R4 : {int(mask_vides.sum())} lignes vides supprimées")

cles_speciales = w[w["sku"].notna() & ~w["sku"].astype(str).str.fullmatch(r"\d+")]["sku"].unique().tolist()
print(f"R7 : clés non numériques conservées : {cles_speciales}")

w["total_sales"] = pd.to_numeric(w["total_sales"], errors="coerce").fillna(0)
n_neg_sales = int((w["total_sales"] < 0).sum())
w["total_sales"] = w["total_sales"].abs().astype(int)
print(f"R8 : {n_neg_sales} ventes négatives -> valeur absolue")

# R6 : clé de jointure texte ; en cas de doublon de clé, on garde la ligne la plus renseignée
w["sku_key"] = w["sku"].astype("string").str.strip()
w["nb_nulls"] = w[COLS_WEB].isnull().sum(axis=1)
n_cles_dup = int(w["sku_key"].duplicated(keep=False).sum())
w = w.sort_values("nb_nulls").drop_duplicates(subset=["sku_key"], keep="first").drop(columns="nb_nulls")
print(f"R6 : {n_cles_dup} lignes en doublon de clé -> {len(w)} lignes conservées")
w.head(3)

R5 : 791 doublons complets supprimés
R4 : 1 lignes vides supprimées
R7 : clés non numériques conservées : ['13127-1', 'bon-cadeau-25-euros']
R8 : 2 ventes négatives -> valeur absolue
R6 : 12 lignes en doublon de clé -> 715 lignes conservées


,sku,total_sales,product_type,post_title,post_name,sku_key
1261,15382,1,Vin,Zind-Humbrecht Riesling Grand Cru Rangen De Th...,zind-humbrecht-riesling-gc-rangen-thann-clos-s...,15382
1,16057,5,Vin,Domaine Pellé Sancerre Rouge La Croix Au Garde...,pelle-sancerre-rouge-la-croix-au-garde-2017,16057
2,14692,5,Vin,Château Fonréaud Bordeaux Blanc Le Cygne 2016,fonreaud-bordeaux-blanc-le-cygne-2016,14692


In [6]:
# --- LIAISON : doublons et clé texte ---
liaison_clean = liaison.drop_duplicates()
liaison_clean["id_web_key"] = liaison_clean["id_web"].astype("string").str.strip()

n_sans_web = int(liaison_clean["id_web_key"].isnull().sum())
print(f"{n_sans_web} produits de l'ERP sans correspondance web (= produits non vendus en ligne)")
liaison_clean.head(3)

91 produits de l'ERP sans correspondance web (= produits non vendus en ligne)


,id_web,product_id,id_web_key
0,15298,3847,15298
1,15296,3849,15296
2,15300,3850,15300


<div style="background-color: RGB(51,165,182);">
<h2 style="margin: auto; padding: 20px; color:#fff; ">5 — Fusion des trois sources et validation finale</h2>
</div>

Fusion en deux temps : `erp` ↔ `liaison` sur `product_id`, puis ↔ `web` sur la clé texte (`id_web` = `sku`).
La jointure sur **clés texte** (nouveauté v2) permet de faire correspondre correctement les identifiants non numériques (`13127-1`, `bon-cadeau-25-euros`) — en v1 ils étaient exclus ou modifiés à la main.

**Ancre de validation** : le chiffre d'affaires web recalculé doit valoir **153 748,10 €**, valeur obtenue en v1 et présentée à la direction. Tant que cette ancre tient, le pipeline v2 est fidèle à la v1 tout en étant reproductible.

In [7]:
df_merged = erp.merge(liaison_clean, on="product_id", how="left")
total = df_merged.merge(w, left_on="id_web_key", right_on="sku_key", how="left")

n_match = int(total["sku_key"].notna().sum())
print(f"{total.shape[0]} produits au total ; {n_match} avec correspondance web ; {total.shape[0] - n_match} vendus en cave uniquement")
total.head(3)

825 produits au total ; 714 avec correspondance web ; 111 vendus en cave uniquement


,product_id,onsale_web,price,stock_quantity,stock_status,purchase_price,id_web,id_web_key,sku,total_sales,product_type,post_title,post_name,sku_key
0,3847,1,24.2,16,instock,12.88,15298,15298,15298,6.0,Vin,Pierre Jean Villa Saint-Joseph Préface 2018,pierre-jean-villa-saint-joseph-preface-2018,15298
1,3849,1,34.3,10,instock,17.54,15296,15296,15296,9.0,Vin,Pierre Jean Villa Saint-Joseph Rouge Tildé 2017,pierre-jean-villa-saint-joseph-tilde-2017,15296
2,3850,1,20.8,0,outofstock,10.64,15300,15300,15300,0.0,Vin,Pierre Jean Villa Crozes-Hermitage Accroche Co...,pierre-jean-villa-croze-hermitage-accroche-coe...,15300


In [8]:
# Validation Pandera du dataset fusionné (règles métier strictes, post-nettoyage)
schema_total = DataFrameSchema({
    "product_id": Column(int, Check.gt(0), unique=True, nullable=False),
    "price": Column(float, Check.gt(0)),
    "purchase_price": Column(float, Check.gt(0)),
    "stock_quantity": Column(int, Check.ge(0)),
    "total_sales": Column(float, Check.ge(0), nullable=True),
}, strict=False)

valider("dataset fusionné", schema_total, total)

[OK] dataset fusionné : 825 lignes conformes au contrat


In [9]:
# Ancre de validation vs v1
total["ca"] = total["price"] * total["total_sales"].fillna(0)
ca_web = total.loc[total["sku_key"].notna(), "ca"].sum()
print(f"Chiffre d'affaires web : {ca_web:,.2f} €")
print(f"Ancre v1             : 153 748,10 €")
print(f"Écart                : {abs(ca_web - 153748.10):.2f} €")

Chiffre d'affaires web : 153,748.10 €
Ancre v1             : 153 748,10 €
Écart                : 0.00 €


<div style="background-color: RGB(51,165,182);">
<h2 style="margin: auto; padding: 20px; color:#fff; ">6 — Prix : z-score (v1) vs IQR</h2>
</div>

En v1, les prix aberrants étaient détectés par **z-score** (> 3 écarts-types de la moyenne). Cette méthode suppose une distribution proche de la loi normale ; or les prix du catalogue sont **asymétriques** (quelques grands crus très chers tirent la moyenne et l'écart-type vers le haut, ce qui « masque » des anomalies intermédiaires).

On compare ici le z-score à l'**écart interquartile (IQR)**, méthode robuste car fondée sur les quartiles — c'est la règle chiffrée derrière la boîte à moustaches.

In [10]:
fig = px.box(total, y=["price", "purchase_price"],
             title="Distribution des prix de vente et des prix d'achat",
             labels={"value": "Montant (€)", "variable": ""})
fig.show()

In [11]:
total["price"].describe().round(2)

count    825.00
mean      32.28
std       26.60
min        5.20
25%       14.50
50%       24.30
75%       42.00
max      225.00
Name: price, dtype: float64

In [12]:
# --- Méthode v1 : z-score ---
mu, sigma = total["price"].mean(), total["price"].std()
total["z_score"] = (total["price"] - mu) / sigma

n_12 = int(((total["z_score"] > 1) & (total["z_score"] <= 2)).sum())
n_23 = int(((total["z_score"] > 2) & (total["z_score"] <= 3)).sum())
n_3 = int((total["z_score"] > 3).sum())
print(f"Moyenne {mu:.2f} €, écart-type {sigma:.2f} €")
print(f"z entre 1 et 2 : {n_12} produits ({n_12/len(total)*100:.2f} %)")
print(f"z entre 2 et 3 : {n_23} produits ({n_23/len(total)*100:.2f} %)")
print(f"z > 3          : {n_3} produits -> outliers v1")

fig = px.histogram(total, x="z_score", nbins=60,
                   title="Distribution des z-scores des prix (seuil v1 : z > 3)")
fig.add_vline(x=3, line_dash="dash", line_color="red")
fig.show()

Moyenne 32.28 €, écart-type 26.60 €
z entre 1 et 2 : 66 produits (8.00 %)
z entre 2 et 3 : 17 produits (2.06 %)
z > 3          : 17 produits -> outliers v1


In [13]:
# --- Méthode robuste : IQR ---
q1, q3 = total["price"].quantile(0.25), total["price"].quantile(0.75)
iqr = q3 - q1
borne_basse, borne_haute = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"Q1 = {q1:.2f} €, Q3 = {q3:.2f} €, IQR = {iqr:.2f} €")
print(f"Sont suspects les prix hors [{borne_basse:.2f} € ; {borne_haute:.2f} €]")

total["outlier_iqr"] = (total["price"] < borne_basse) | (total["price"] > borne_haute)
total["outlier_z3"] = total["z_score"] > 3
print(f"Outliers IQR : {int(total['outlier_iqr'].sum())} produits")
print(f"Outliers z>3 : {int(total['outlier_z3'].sum())} produits")

Q1 = 14.50 €, Q3 = 42.00 €, IQR = 27.50 €
Sont suspects les prix hors [-26.75 € ; 83.25 €]
Outliers IQR : 36 produits
Outliers z>3 : 17 produits


### Comparaison des deux listes

Chaque écart entre les deux méthodes est examiné produit par produit (critère d'acceptation du cahier des charges).

In [14]:
# Produits détectés par IQR mais pas par le z-score (les « angles morts » de la v1)
angles_morts = total[total["outlier_iqr"] & ~total["outlier_z3"]].sort_values("price", ascending=False)
print(f"{len(angles_morts)} produits détectés par l'IQR mais invisibles au z-score :")
display(angles_morts[["product_id", "post_title", "price", "z_score"]].head(20))

# Et l'inverse ?
print("Détectés par z>3 mais pas par l'IQR :",
      int((total["outlier_z3"] & ~total["outlier_iqr"]).sum()))

19 produits détectés par l'IQR mais invisibles au z-score :


,product_id,post_title,price,z_score
475,5025,Champagne Agrapart &amp; Fils L'Avizoise Extra...,112.0,2.996721
240,4582,Château de Meursault Puligny-Montrachet 1er Cr...,109.6,2.906507
228,4404,Cognac Frapin Château de Fontpinot XO,108.5,2.865158
702,6201,David Duband Chambolle-Musigny 1er Cru Les Sen...,105.6,2.756149
466,5008,Domaine des Comtes Lafon Volnay 1er Cru Santen...,105.0,2.733595
465,5007,Domaine des Comtes Lafon Volnay 1er Cru Santen...,105.0,2.733595
231,4407,Cognac Frapin Cigar Blend,104.0,2.696006
410,4903,Domaine Des Croix Corton Grand Cru Les Grèves ...,102.3,2.632103
66,4115,Zind-Humbrecht Riesling Grand Cru Rangen De Th...,100.0,2.545648
710,6214,Domaine des Comtes Lafon Volnay 1er Cru Champa...,99.0,2.508058


Détectés par z>3 mais pas par l'IQR : 0


In [15]:
fig = px.scatter(total, x="price", y="z_score",
                 color=total["outlier_iqr"].map({True: "Outlier IQR", False: "OK"}),
                 hover_data=["product_id", "post_title"],
                 title="Prix vs z-score — les outliers IQR (rouge) ne sont pas tous au-delà de z = 3")
fig.add_hline(y=3, line_dash="dash", line_color="gray")
fig.show()

**Lecture métier.** Les 19 produits « angles morts » sont des **grands crus et cognacs premium** (85 € à 110 €) : chers, mais pas assez extrêmes pour dépasser z = 3 sur une distribution tirée vers le haut par les produits à 150-225 €. L'IQR, lui, les signale. Pour un gestionnaire de stock, ce sont précisément les références à vérifier (prix de vente cohérent avec le prix d'achat ? stock immobilisé élevé ?).

**Décision documentée (journal IA)** : l'IQR devient le détecteur de premier niveau ; le z-score est conservé à titre pédagogique et de continuité avec la v1.

<div style="background-color: RGB(51,165,182);">
<h2 style="margin: auto; padding: 20px; color:#fff; ">7 — Détection multivariée : Isolation Forest</h2>
</div>

Le z-score et l'IQR analysent **une variable à la fois**. Or certaines anomalies ne sont visibles qu'en **croisant** plusieurs variables : un produit au prix de vente normal peut avoir un **prix d'achat supérieur au prix de vente** (marge négative) — invisible pour les méthodes univariées sur le prix.

L'**Isolation Forest** (scikit-learn, non supervisé) isole les points par découpes aléatoires : une anomalie est isolée en peu de coupes. On l'applique à 4 variables : `price`, `purchase_price`, `stock_quantity`, `total_sales`.

**Précautions** : seed fixée (`SEED = 42`, section 1) pour la reproductibilité ; proportion d'anomalies attendue fixée à 2 % (paramètre métier) ; les anomalies détectées sont **relues avec un regard métier** avant tout usage — l'algorithme signale, il ne décide pas.

In [16]:
features = ["price", "purchase_price", "stock_quantity", "total_sales"]
X = total[features].fillna(0)

iso = IsolationForest(contamination=0.02, random_state=SEED)
total["anomalie_if"] = iso.fit_predict(X) == -1

anomalies = total[total["anomalie_if"]].copy()
anomalies["marge"] = anomalies["price"] - anomalies["purchase_price"]
anomalies = anomalies.sort_values("marge")
print(f"{len(anomalies)} anomalies détectées ({len(anomalies)/len(total)*100:.1f} % du catalogue)")
display(anomalies[["product_id", "post_title", "price", "purchase_price", "marge",
                   "stock_quantity", "total_sales"]].head(10))

17 anomalies détectées (2.1 % du catalogue)


,product_id,post_title,price,purchase_price,marge,stock_quantity,total_sales
210,4355,Champagne Egly-Ouriet Grand Cru Blanc de Noirs,12.65,77.48,-64.83,97,0.0
724,6324,Pierre Jean Villa Côte Rôtie Fongeant 2017,92.00,99.00,-7.00,18,17.0
393,4867,Château De La Selve IGP Coteaux de l'Ardèche M...,9.90,4.86,5.04,121,36.0
79,4150,Champagne Mailly Grand Cru Intemporelle 2010,59.00,35.45,23.55,123,116.0
207,4350,Champagne Egly-Ouriet Grand Cru Extra Brut V.P.,79.50,47.30,32.20,145,7.0
209,4353,Champagne Egly-Ouriet Grand Cru Brut Rosé,79.50,45.91,33.59,127,14.0
205,4337,Champagne Mailly Grand Cru Les Echansons 2007,83.00,48.90,34.10,145,0.0
476,5026,Champagne Agrapart &amp; Fils Minéral Extra Br...,86.80,50.13,36.67,101,9.0
475,5025,Champagne Agrapart &amp; Fils L'Avizoise Extra...,112.00,68.60,43.40,136,6.0
697,6126,Champagne Gosset Célébris Vintage 2007,135.00,80.33,54.67,138,5.0


In [17]:
# Validation métier : l'algorithme retrouve-t-il les marges négatives ?
total["taux_marge"] = (total["price"] - total["purchase_price"]) / total["price"]
marges_neg = total[total["taux_marge"] < 0]
print("Produits à marge négative :")
display(marges_neg[["product_id", "post_title", "price", "purchase_price",
                    "taux_marge", "anomalie_if"]])

fig = px.scatter(total, x="purchase_price", y="price",
                 color=total["anomalie_if"].map({True: "Anomalie IF", False: "OK"}),
                 hover_data=["product_id", "post_title"],
                 title="Prix de vente vs prix d'achat — sous la diagonale : marge négative")
fig.add_shape(type="line", x0=0, y0=0, x1=240, y1=240, line=dict(dash="dash", color="gray"))
fig.show()

Produits à marge négative :


,product_id,post_title,price,purchase_price,taux_marge,anomalie_if
210,4355,Champagne Egly-Ouriet Grand Cru Blanc de Noirs,12.65,77.48,-5.124901,True
391,4864,NaN,8.30,9.99,-0.203614,False
724,6324,Pierre Jean Villa Côte Rôtie Fongeant 2017,92.00,99.00,-0.076087,True
817,7196,Pierre Jean Villa Côte Rôtie Fongeant 2017,31.00,31.20,-0.006452,False


**Résultat clé.** L'Isolation Forest détecte les marges négatives **franches** (ex. product_id 4355 : prix de vente 12,65 € pour un prix d'achat de 77,48 € — très probablement une inversion de saisie) et signale aussi des profils atypiques cohérents (fort stock + fortes ventes, produits premium). Les marges légèrement négatives (−0,20 € à −1,69 €) ne sont pas détectées : l'écart est trop faible pour isoler ces produits. C'est une **limite documentée** : l'IF complète les règles métier, elle ne les remplace pas — le contrôle `taux_marge < 0` reste une règle explicite de la section 9.

<div style="background-color: RGB(51,165,182);">
<h2 style="margin: auto; padding: 20px; color:#fff; ">8 — Performance mesurée : pandas vs Polars vs DuckDB</h2>
</div>

La veille (axe 3) identifiait Polars (moteur Rust multi-thread) et DuckDB (SQL embarqué) comme alternatives à pandas. Plutôt que de décider sur le papier, on **chronomètre** l'opération centrale du notebook — les deux jointures + le calcul du CA — dans les trois outils, sur les données réelles déjà nettoyées (sections 4 et 5). 5 exécutions chacune, moyenne retenue. Critère de décision : le gain doit être **significatif** pour justifier une réécriture ; sinon, la sobriété commande de garder pandas.

In [18]:
import time
import polars as pl
import duckdb

def pipeline_pandas():
    m = erp.merge(liaison_clean, on="product_id", how="left")
    t = m.merge(w, left_on="id_web_key", right_on="sku_key", how="left")
    t["ca"] = t["price"] * t["total_sales"].fillna(0)
    return float(t.loc[t["sku_key"].notna(), "ca"].sum())

erp_pl = pl.from_pandas(erp)
liaison_pl = pl.from_pandas(liaison_clean[["product_id", "id_web_key"]])
w_pl = pl.from_pandas(w[["sku_key", "total_sales"]].assign(total_sales=w["total_sales"].astype(float)))

def pipeline_polars():
    m = erp_pl.join(liaison_pl, on="product_id", how="left")
    t = m.join(w_pl, left_on="id_web_key", right_on="sku_key", how="left")
    t = t.with_columns((pl.col("price") * pl.col("total_sales").fill_null(0)).alias("ca"))
    return float(t.filter(pl.col("total_sales").is_not_null())["ca"].sum())

con = duckdb.connect()
con.register("erp_tbl", erp)
con.register("liaison_tbl", liaison_clean[["product_id", "id_web_key"]])
con.register("web_tbl", w[["sku_key", "total_sales"]])
QUERY = """
SELECT SUM(e.price * COALESCE(w.total_sales, 0))
FROM erp_tbl e
LEFT JOIN liaison_tbl l USING (product_id)
LEFT JOIN web_tbl w ON l.id_web_key = w.sku_key
WHERE w.sku_key IS NOT NULL
"""

def pipeline_duckdb():
    return float(con.execute(QUERY).fetchone()[0])

resultats = {}
for nom, fct in [("pandas", pipeline_pandas), ("Polars", pipeline_polars), ("DuckDB", pipeline_duckdb)]:
    temps, ca = [], None
    for _ in range(5):
        t0 = time.perf_counter()
        ca = fct()
        temps.append(time.perf_counter() - t0)
    resultats[nom] = (round(sum(temps) / len(temps) * 1000, 1), round(ca, 2))

bench = pd.DataFrame(resultats, index=["Temps moyen (ms)", "CA web (EUR)"]).T
bench["Rapport vs pandas"] = (bench["Temps moyen (ms)"] / bench.loc["pandas", "Temps moyen (ms)"]).round(2)
bench

,Temps moyen (ms),CA web (EUR),Rapport vs pandas
pandas,4.4,153748.1,1.00
Polars,3.0,153748.1,0.68
DuckDB,18.7,153748.1,4.25


**Résultat et décision.** Les trois outils produisent **exactement le même CA** (153 748,10 € — l'ancre tient partout). Sur ces volumes (825 produits × 714 lignes web rattachées), l'opération complète prend quelques **millisecondes** dans les trois cas : l'écart, même s'il est favorable à Polars/DuckDB en relatif, est sans conséquence pratique — le notebook complet s'exécute en moins d'une minute.

**Décision documentée (journal IA)** : pandas est **conservé**. Réécrire le pipeline n'apporterait rien de mesurable au métier, ajouterait un risque de régression et un coût de formation — une décision de **sobriété** (ne pas réécrire ce qui fonctionne). Réévaluation planifiée si les volumes dépassent environ 100 000 lignes par export : le code du benchmark ci-dessus servira alors de test de référence.

<div style="background-color: RGB(51,165,182);">
<h2 style="margin: auto; padding: 20px; color:#fff; ">9 — Analyses métier</h2>
</div>

Cette section reprend les analyses de la v1 (inchangées dans leur logique, désormais adossées au pipeline validé) : chiffre d'affaires, Pareto, quantités vendues, rotation des stocks, marges, classification ABC et flags d'action.

In [19]:
# Chiffre d'affaires
ca_total = total["ca"].sum()
ca_web = total.loc[total["sku_key"].notna(), "ca"].sum()
print(f"CA total (web) : {ca_web:,.2f} €")

top20_ca = total.nlargest(20, "ca")
fig = px.bar(top20_ca, x="post_title", y="ca",
             title="Top 20 des produits par chiffre d'affaires",
             labels={"ca": "CA (€)", "post_title": "Produit"})
fig.update_layout(xaxis_tickangle=-45)
fig.show()

CA total (web) : 153,748.10 €


In [20]:
# Pareto : combien de produits font 80 % du CA ?
total_sorted = total.sort_values("ca", ascending=False).reset_index(drop=True)
total_sorted["ca_cum_pct"] = total_sorted["ca"].cumsum() / ca_total * 100
n_80 = int((total_sorted["ca_cum_pct"] <= 80).sum()) + 1
print(f"{n_80} produits représentent 80 % du CA ({n_80/len(total)*100:.2f} % du catalogue)")

fig = px.line(total_sorted.reset_index(), x="index", y="ca_cum_pct",
              title="Courbe de Pareto du chiffre d'affaires",
              labels={"index": "Nombre de produits (triés par CA décroissant)", "ca_cum_pct": "CA cumulé (%)"})
fig.add_hline(y=80, line_dash="dash", line_color="red")
fig.show()

446 produits représentent 80 % du CA (54.06 % du catalogue)


In [21]:
# Quantités vendues
ventes = total["total_sales"].fillna(0)
print(f"Total de produits vendus : {int(ventes.sum())}")
print(f"Moyenne par produit : {ventes.mean():.2f}")

top20_q = total.nlargest(20, "total_sales")
fig = px.bar(top20_q, x="post_title", y="total_sales",
             title="Top 20 des produits par quantité vendue",
             labels={"total_sales": "Quantité vendue", "post_title": "Produit"})
fig.update_layout(xaxis_tickangle=-45)
fig.show()

Total de produits vendus : 7628
Moyenne par produit : 9.25


### Rotation des stocks

La formule classique (ventes / stock moyen) n'est pas applicable : nous n'avons pas le stock moyen sur la période. On utilise comme en v1 la **lenteur de vente** = `stock_quantity / total_sales` — plus le chiffre est élevé, plus le produit se vend lentement (calculée uniquement pour les produits vendus en ligne).

In [22]:
total["rotation_stock"] = np.where(
    total["total_sales"].fillna(0) > 0,
    total["stock_quantity"] / total["total_sales"],
    np.nan,
)
rot = total["rotation_stock"].dropna()
print(f"Lenteur de vente : min {rot.min():.2f}, médiane {rot.median():.2f}, max {rot.max():.2f}")

valorisation = (total["stock_quantity"] * total["purchase_price"]).sum()
print(f"Valorisation totale des stocks : {valorisation:,.2f} €")
print(f"Nombre total de produits en stock : {int(total['stock_quantity'].sum())}")

flop20 = total[total["rotation_stock"].notna()].nlargest(20, "rotation_stock")
fig = px.bar(flop20, x="post_title", y="rotation_stock",
             title="FLOP 20 — les produits qui se vendent le plus lentement",
             labels={"rotation_stock": "Lenteur de vente (stock / ventes)", "post_title": "Produit"})
fig.update_layout(xaxis_tickangle=-45)
fig.show()

Lenteur de vente : min 0.00, médiane 2.29, max 31.25
Valorisation totale des stocks : 298,627.66 €
Nombre total de produits en stock : 17822


In [23]:
# Taux de marge
tm = total["taux_marge"].dropna()
print(f"Taux de marge : min {tm.min():.2%}, max {tm.max():.2%} (hors marges négatives : min {tm[tm >= 0].min():.2%})")
print(f"Produits à marge négative : {len(marges_neg)} (voir section 7)")

marge_type = (total[total["taux_marge"] >= 0]
              .groupby("product_type")["taux_marge"].mean().sort_values(ascending=False).reset_index())
fig = px.bar(marge_type, x="product_type", y="taux_marge",
             title="Taux de marge moyen par type de produit (hors marges négatives)",
             labels={"taux_marge": "Taux de marge moyen", "product_type": "Type de produit"})
fig.show()

Taux de marge : min -512.49%, max 56.46% (hors marges négatives : min 35.65%)
Produits à marge négative : 4 (voir section 7)


### Classification ABC et flags d'action

- **A** : produits représentant jusqu'à 80 % du CA ; **B** : de 80 % à 95 % ; **C** : au-delà de 95 %.
- **REORDER** : classe A, lenteur de vente ≤ 1,5 (rotation très rapide) et marge positive → risque de rupture.
- **PROMO** : classe B ou C et lenteur de vente > 4 → stock qui dort.
- **CHECK** : anomalie détectée (Isolation Forest, section 7) ou marge négative → vérification manuelle.

In [24]:
# Classification ABC sur le CA
total_sorted["ABC"] = pd.cut(total_sorted["ca_cum_pct"], bins=[0, 80, 95, 100],
                             labels=["A", "B", "C"], include_lowest=True)
total = total.merge(total_sorted[["product_id", "ABC"]], on="product_id", how="left")

conditions = [
    (total["ABC"] == "A") & (total["rotation_stock"] <= 1.5) & (total["taux_marge"] >= 0),
    (total["ABC"].isin(["B", "C"])) & (total["rotation_stock"] > 4),
    (total["anomalie_if"]) | (total["taux_marge"] < 0),
]
total["action_flag"] = np.select(conditions, ["reorder", "promo", "check"], default="")

print(total["action_flag"].value_counts().to_string())
print(f"\nRépartition ABC : {total['ABC'].value_counts().to_dict()}")

action_flag
           641
reorder    164
check       15
promo        5



Répartition ABC : {'A': 445, 'B': 222, 'C': 158}


In [25]:
# Corrélations
corr = total[["stock_quantity", "total_sales", "price"]].corr().round(2)
fig = px.imshow(corr, text_auto=True, color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
                title="Corrélations stock / ventes / prix")
fig.show()

In [26]:
# Exports des livrables
import os
os.makedirs("sortie", exist_ok=True)
total.to_csv("sortie/dataset_final.csv", index=False)
for flag in ["reorder", "promo", "check"]:
    total[total["action_flag"] == flag].to_csv(f"sortie/{flag}.csv", index=False)
total[total["outlier_iqr"]].to_csv("sortie/outliers_prix_iqr.csv", index=False)
anomalies.to_csv("sortie/anomalies_isolation_forest.csv", index=False)
print("Exports écrits dans sortie/ :", os.listdir("sortie"))

Exports écrits dans sortie/ : ['anomalies_isolation_forest.csv', 'check.csv', 'dataset_final.csv', 'outliers_prix_iqr.csv', 'promo.csv', 'reorder.csv']


<div style="background-color: RGB(51,165,182);">
<h2 style="margin: auto, padding: 20px; color:#fff; ">10 — Synthèse, limites et recommandations</h2>
</div>

### Résultats
- **Pipeline reproductible** : le CA web recalculé (153 748,10 €) reproduit **à l'euro près** le chiffre présenté en v1, avec un nettoyage désormais fondé sur des règles déclaratives (Pandera) et non sur des corrections manuelles.
- **Détection d'anomalies robustifiée** : l'IQR signale 36 produits (contre 17 au z-score), dont 19 « angles morts » de la v1 — des grands crus premium à vérifier. L'Isolation Forest détecte les anomalies **multivariées**, dont la marge négative franche du product_id 4355.
- **Livrables actionnables** : listes REORDER (risque de rupture), PROMO (stock dormant), CHECK (à vérifier) exportées en CSV.

### Limites
- La lenteur de vente (stock / ventes) n'est pas la rotation classique : le stock moyen sur la période n'est pas disponible.
- L'Isolation Forest dépend du paramètre `contamination` (2 %) et ne détecte pas les marges légèrement négatives : les règles métier explicites restent nécessaires.
- Les ventes négatives (retours) sont passées en valeur absolue : hypothèse à confirmer avec l'équipe e-commerce.

### Recommandations
1. Corriger en priorité les produits **CHECK** (marges négatives = erreurs de saisie probables).
2. Réapprovisionner les **REORDER** avant rupture ; envisager des promotions sur les **PROMO**.
3. Rejouer ce notebook à chaque nouvel export : le contrat Pandera signalera immédiatement toute dérive de qualité.

### Pistes d'amélioration continue (veille)
- **Great Expectations** si les contrôles qualité doivent produire des rapports HTML réguliers pour la direction.
- **Polars / DuckDB** si les volumes augmentent fortement (gain marginal aux volumes actuels — décision mesurée et documentée).

<div style="background-color: RGB(51,165,182);">
<h2 style="margin: auto; padding: 20px; color:#fff; ">11 — Reproductibilité</h2>
</div>

**Pour relancer ce notebook :**

1. Créer un environnement : `pip install -r requirements.txt`
2. Placer les exports bruts dans `data/` (`erp.csv`, `web.csv`, `liaison.csv`) — ne jamais les modifier.
3. Exécuter toutes les cellules dans l'ordre (Kernel → Restart & Run All). Durée : moins d'une minute.
4. Les livrables sont régénérés dans `sortie/`.

**Garanties :** versions figées (section 1) · seed unique `SEED = 42` · aucune modification manuelle des données · chaque correction est une règle documentée (section 4) · ancre de validation : CA web = 153 748,10 € (section 5).